In [1]:
import itertools as it
import os
import sys
from copy import deepcopy
from glob import glob
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap

import joblib
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_curve, roc_auc_score
from plotnine import *

from nilearn.plotting import plot_anat, plot_stat_map

sys.path.append("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code")
from project_utils import *

In [2]:
wdir = "/ceph/chpc/shared/aristeidis_sotiras_group/b.y.yang_scratch/BradenADLongitudinalPrediction/3_TrainTestModel/simplified_experiments/base"

# Get feature names

In [11]:
model_dict = joblib.load(os.path.join(wdir, "avn__svm__5yr__site.joblib"))  # load example model
feature_names = model_dict["trained_models"]["A4"]["model"].best_estimator_[:-1].get_feature_names_out()  # get feature names

# amyloid features and feature names
amyloid_idx = np.char.endswith(feature_names.astype(str), ".amyloid")
feature_names_amyloid = feature_names[amyloid_idx].astype(str)
feature_names_amyloid_raw = feature_names_amyloid
for sub in ["amyloid__", ".amyloid"]:
    feature_names_amyloid_raw = np.char.replace(feature_names_amyloid_raw, sub, "")

# volume features and feature names
volume_idx = np.char.endswith(feature_names.astype(str), ".vol")
feature_names_volume = feature_names[volume_idx].astype(str)
feature_names_volume_raw = feature_names_volume
for sub in ["volume__", ".vol"]:
    feature_names_volume_raw = np.char.replace(feature_names_volume_raw, sub, "")

# nonimg features and feature names
nonimg_idx = np.char.startswith(feature_names.astype(str), "nonimg__")
feature_names_nonimg = feature_names[nonimg_idx].astype(str)
feature_names_nonimg_raw = feature_names_nonimg
for sub in ["nonimg__"]:
    feature_names_nonimg_raw = np.char.replace(feature_names_nonimg_raw, sub, "")

In [32]:
models_site = {t: joblib.load(os.path.join(wdir, f"avn__svm__{t}yr__site.joblib")) for t in [1,2,3,4,5]}
models_tracer = {t: joblib.load(os.path.join(wdir, f"avn__svm__{t}yr__tracer.joblib")) for t in [1,2,3,4,5]}

In [71]:
# compute pairwise differences of Haufe weights across time windows
haufe_arr_site = {}
haufe_diff_site = {}
for site in models_site[1]["trained_models"].keys():
    haufe_arr = np.concatenate([m["trained_models"][site]["haufe"] for m in models_site.values()], axis = 1)
    haufe_diff = np.stack([haufe_arr[:,j] - haufe_arr[:,i] for i in range(5) for j in range(i+1, 5)], axis = 1)
    haufe_arr_site[site] = haufe_arr; haufe_diff_site[site] = haufe_diff

haufe_arr_tracer = {}
haufe_diff_tracer = {}
for tracer in models_tracer[1]["trained_models"].keys():
    haufe_arr = np.concatenate([m["trained_models"][tracer]["haufe"] for m in models_tracer.values()], axis = 1)
    haufe_diff = np.stack([haufe_arr[:,j] - haufe_arr[:,i] for i in range(5) for j in range(i+1, 5)], axis = 1)
    haufe_arr_tracer[tracer] = haufe_arr; haufe_diff_tracer[tracer] = haufe_diff

# Haufe coefficients

In [105]:
haufe_df_list = []
for site in haufe_arr_site.keys():
    df = pd.DataFrame(haufe_arr_site[site], index = feature_names, columns = [1,2,3,4,5]).reset_index().rename(columns = {"index": "feature_name"})
    df["leave_out_group"] = site
    haufe_df_list.append(df)
for tracer in haufe_arr_tracer.keys():
    df = pd.DataFrame(haufe_arr_tracer[tracer], index = feature_names, columns = [1,2,3,4,5]).reset_index().rename(columns = {"index": "feature_name"})
    df["leave_out_group"] = tracer
    haufe_df_list.append(df)

In [113]:
haufe_df = pd.concat(haufe_df_list).reset_index(drop = True)
haufe_df["feature_type"] = haufe_df["feature_name"].str.split("__", expand = True)[0]

haufe_long = haufe_df.melt(id_vars = ["feature_name", "feature_type", "leave_out_group"], var_name = "time_window", value_name = "haufe")
haufe_long.to_csv("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/4_FeatureImportance/haufe.csv")

# Change in Haufe feature importance across time windows

In [66]:
# difference in Haufe across all models (leave-out site + leave-out tracer)
haufe_diff_all = np.concatenate([h for h in haufe_diff_site.values()] + [h for h in haufe_diff_tracer.values()], axis = 1)
haufe_diff_all_df = pd.DataFrame(haufe_diff_all, index = feature_names).reset_index().rename(columns = {"index": "feature_name"})
haufe_diff_all_df["feature_type"] = haufe_diff_all_df["feature_name"].str.split("__", expand = True)[0]

haufe_diff_all_long = haufe_diff_all_df.melt(id_vars = ["feature_name", "feature_type"], var_name = "idx", value_name = "haufe")
haufe_diff_all_long.to_csv("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/4_FeatureImportance/haufe_diff.csv")

# Slope of Haufe feature importance

Here, we take the absolute value of the Haufe coefficient before fitting a linear regression

In [118]:
slopes_site = np.array([[np.polyfit([1,2,3,4,5], row, 1)[0] for row in np.abs(haufe_arr_site[site])] for site in haufe_arr_site.keys()])
slopes_tracer = np.array([[np.polyfit([1,2,3,4,5], row, 1)[0] for row in np.abs(haufe_arr_tracer[tracer])] for tracer in haufe_arr_tracer.keys()])

In [119]:
slopes_all = np.concatenate([slopes_site, slopes_tracer], axis = 0).T

In [ ]:
slopes_df = pd.DataFrame(slopes_all, index = feature_names).reset_index().rename(columns = {"index": "feature_name"})
slopes_df["feature_type"] = slopes_df["feature_name"].str.split("__", expand = True)[0]

slopes_long = slopes_df.melt(id_vars = ["feature_name", "feature_type"], var_name = "idx", value_name = "slope")
slopes_long.to_csv("/home/b.y.yang/sotiraslab/BradenADLongitudinalPrediction/code/4_FeatureImportance/haufe_slope.csv")